In [2]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

# 输入4维 → 隐藏3维 → 输出2类
X = torch.randn(5, 4)
y = torch.tensor([0, 1, 1, 0, 1])

W1 = torch.randn(4, 3, requires_grad=True)
b1 = torch.zeros(3, requires_grad=True)
W2 = torch.randn(3, 2, requires_grad=True)
b2 = torch.zeros(2, requires_grad=True)

# 前向：z1 -> a1 -> logits
z1 = X @ W1 + b1
a1 = torch.relu(z1)
logits = a1 @ W2 + b2

In [3]:
loss = F.cross_entropy(logits, y)
loss.backward()

gW2_auto = W2.grad.clone()
gW1_auto = W1.grad.clone()
print("自动 | dL/dW2 首元素:", gW2_auto.flatten()[0].item())
print("自动 | dL/dW1 首元素:", gW1_auto.flatten()[0].item())

自动 | dL/dW2 首元素: 0.08125960826873779
自动 | dL/dW1 首元素: 0.09501031041145325


In [4]:
N = X.shape[0]

# --- 输出层 ---
# 公式：dL/dlogits = (softmax(logits) - onehot) / N
probs = torch.softmax(logits, dim=1)
onehot = F.one_hot(y, num_classes=2).float()
dlogits = (probs - onehot) / N          # 形状 [5, 2]，误差传播的起点

# --- 反传到 W2, b2 ---
# 公式：dL/dW2 = a1ᵀ · dlogits ；dL/db2 = dlogits 按样本求和
gW2_manual = a1.T @ dlogits              # [3,5]@[5,2] -> [3,2] ✓与W2同形
gb2_manual = dlogits.sum(dim=0)          # [2]

# --- 误差继续回传到隐藏层 ---
# 公式：δ_hidden = (dlogits · W2ᵀ) * ReLU'(z1)，ReLU'(z1)=(z1>0)
d_a1 = dlogits @ W2.T                    # 先过 W2，[5,3]
d_z1 = d_a1 * (z1 > 0)                   # 再过 ReLU 导数，逐元素相乘

# --- 反传到 W1, b1 ---
# 公式：dL/dW1 = Xᵀ · δ_hidden ；dL/db1 = δ_hidden 按样本求和
gW1_manual = X.T @ d_z1                  # [4,5]@[5,3] -> [4,3] ✓与W1同形
gb1_manual = d_z1.sum(dim=0)             # [3]
print("手动 dL/dW2 首元素:", gW2_manual.flatten()[0].item())
print("手动 dL/dW1 首元素:", gW1_manual.flatten()[0].item())

手动 dL/dW2 首元素: 0.08125962316989899
手动 dL/dW1 首元素: 0.09501028060913086


In [5]:
diff_w2 = (gW2_manual - gW2_auto).abs().max().item()
diff_w1 = (gW1_manual - gW1_auto).abs().max().item()
print(f"W2 最大误差: {diff_w2:.2e}")
print(f"W1 最大误差: {diff_w1:.2e}")
print("对拍", "一致 ✅" if max(diff_w1, diff_w2) < 1e-6 else "不一致 ❌")

W2 最大误差: 2.98e-08
W1 最大误差: 5.96e-08
对拍 一致 ✅
